<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 11 · Yield-Curve Dynamics and Factor Models

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch11_yield_curve_dynamics_factor_models.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Yield curves move every day, and risk work starts from changes rather than levels. In this notebook you convert a frozen curve history into basis-point changes, reproduce the chapter's PCA decomposition of those changes, and read a Nelson-Siegel fit on the same maturity grid. You learn to identify which statistical factor behaves like level, slope, or curvature; to state the variance share each component explains; and to treat factor labels as empirical diagnostics that the loading signs, not the names, must justify. You also see how the official ECB AAA yield-curve snapshot in the companion repository extends the same workflow to real market data.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script reads the frozen yield-curve history and prints daily changes in basis points for the 1Y, 2Y, 5Y, and 10Y tenors. It then runs PCA on the covariance of those changes, reporting variance shares and loadings for the first three components, and finally prints a Nelson-Siegel curve fitted to the same four maturities.

In [ ]:
SCRIPT = ROOT / "code" / "ch11_yield_curve_dynamics_factor_models.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Work through the printed tables before writing any curve commentary.

- Confirm the units: yields are decimals and the change table is in basis points because the script scales decimal differences by 10,000.
- Check the explained-variance column: the first component explains about 99% of the variance in this teaching sample, so the sample is nearly one-factor.
- Read the PC1 loadings: all four tenors share the same sign, so it is level-like; PC2 contrasts short and long maturities, so it is slope-like.
- Treat the Nelson-Siegel fit as a parametric shape summary with fixed parameters (beta0 0.045, beta1 -0.015, beta2 -0.010, tau 2.5), not as a fit of the PCA factors.
- Remember the history is a teaching snapshot; the official ECB AAA file in the data folder is the real-world extension.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch11_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Level, slope, and curvature from three yields

The chapter script decomposes observed changes; this mini-lab builds the mechanism from the other side. Daily changes at the 2Y, 5Y, and 10Y tenors are constructed from a common level shock, a short-versus-long tilt, and a wings-versus-belly shock, all drawn from a seeded generator and expressed in basis points. PCA of the resulting 3x3 covariance should then recover exactly those three shapes, and the explained-variance shares show how strongly each is represented in the sample.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)  # deterministic seed
z = rng.standard_normal((24, 3))  # factor shocks
changes = pd.DataFrame({  # daily yield changes in bp
    "2y": 16 * z[:, 0] - 5 * z[:, 1] + 2 * z[:, 2],
    "5y": 16 * z[:, 0] - 4 * z[:, 2],
    "10y": 16 * z[:, 0] + 5 * z[:, 1] + 2 * z[:, 2],
})
cov = np.cov(changes, rowvar=False)  # 3x3 covariance
values, vectors = np.linalg.eigh(cov)  # eigensystem
order = np.argsort(values)[::-1]  # descending variance
loadings = pd.DataFrame(vectors[:, order], index=changes.columns)
print(loadings.round(3))  # PC1 level, PC2 slope, PC3 curvature
print("explained:", (values[order] / values.sum()).round(3))

### What to observe

PC1 loads with the same sign on all three tenors (-0.527, -0.594, -0.608) and explains 87.9% of the variance: level. PC2 contrasts the 2Y and 10Y ends (-0.672 versus 0.726) and explains 8.3%: slope. PC3 loads on the wings against the belly (0.521, -0.791, 0.322) and explains 3.8%: curvature. With a different seed, the slope and curvature shares could swap places.

## Exercises

1. **Run PCA on the official snapshot.** Load the ECB AAA yield-curve file from the data folder, convert the tenor columns to basis-point changes, and compute the first three components. *Hint:* Follow the same covariance-and-eigensystem steps as the chapter script, and compare the explained shares with the nearly one-factor teaching sample.
2. **Classify each component.** For the official sample, label the first three components as level, slope, or curvature using only the loading signs. *Hint:* Same sign across tenors means level-like; opposite signs at the short and long end mean slope-like; a sign flip in the belly means curvature-like.
3. **Interpret the Nelson-Siegel parameters.** With beta0 = 0.045, beta1 = -0.015, beta2 = -0.010, and tau = 2.5, explain what each parameter contributes at the 1Y and 10Y points of the fitted curve. *Hint:* The slope loading (1 - exp(-T/tau))/(T/tau) starts near 1 at short T and decays toward 0 as T grows.
4. **Stress one factor.** Add 25 basis points to every yield level in the teaching history and rerun the PCA. *Hint:* The decomposition is computed on changes, so a constant level shift leaves components and variance shares unchanged; explain why.
5. **Compare change-based and level-based PCA.** Redo the decomposition on yield levels instead of changes and discuss how the interpretation differs. *Hint:* Levels are highly persistent, so the first component mechanically captures the common level of yields rather than the way the curve moves.

For more detail, see Chapter 11 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">